In [2]:
# MiniLM-Bench: Google Colab Training Guide
#
# This script trains all 8 attention variants on Colab with:
# - Google Drive persistence (survives disconnections)
# - Automatic checkpoint resume
# - Optimized settings for T4 (free) or A100 (Colab Pro)
# - Sequential training with progress tracking
#
# Usage:
# 1. Upload minilm-bench/ folder to Google Drive
# 2. Open this as a Colab notebook or run the cells sequentially
# 3. Training auto-resumes on reconnection

# ============================================================
# Cell 1: Mount Drive + Setup
# ============================================================

import os
import subprocess

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive/')

# Project paths — ALL state lives on Drive
DRIVE_PROJECT = "/content/drive/MyDrive/minilm-bench"
DRIVE_DATA = f"{DRIVE_PROJECT}/data/tokenized"
DRIVE_CHECKPOINTS = f"{DRIVE_PROJECT}/checkpoints"
DRIVE_RESULTS = f"{DRIVE_PROJECT}/results"

# Clone or sync project
if not os.path.exists(f"{DRIVE_PROJECT}/model"):
    print("ERROR: Upload minilm-bench/ to Google Drive first!")
    print("Expected path: My Drive/minilm-bench/")
    raise FileNotFoundError(f"{DRIVE_PROJECT}/model not found")

# Work from Drive directly (persistence!)
os.chdir(DRIVE_PROJECT)

# Install dependencies
subprocess.run(["pip", "install", "-q", "torch", "numpy", "tiktoken", "pyyaml", "wandb"], check=True)

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

Mounted at /content/drive/
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Memory: 102.0 GB
BF16 support: True


In [3]:
# ============================================================
# Cell 2: Detect GPU + Set Optimal Config
# ============================================================

gpu_name = torch.cuda.get_device_name(0).lower()
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

# Auto-configure based on GPU type
# if "a100" in gpu_name:
    # CONFIG = {
    #     "batch_size": 128,        # was 32 → fill GPU memory
    #     "grad_accum_steps": 1,    # was 4 → no accumulation needed
    #     "d_model": 768,
    #     "n_layers": 12,
    #     "max_steps": 20000,
    #     "eval_interval": 1000,
    #     "checkpoint_interval": 2000,
    #     "compile": True,
    # }
    #print("🚀 A100 detected — using FULL MEMORY config")
if "a100" in gpu_name or gpu_mem_gb > 70:
    CONFIG = {
        "batch_size": 64,
        "grad_accum_steps": 2,    # 64 × 2 = 128 effective (same as before)
        "d_model": 512,
        "n_layers": 8,
        "max_steps": 5000,
        "eval_interval": 500,
        "checkpoint_interval": 1000,
        "compile": False,
    }

    print(f"🚀 High-end GPU detected ({gpu_name}, {gpu_mem_gb:.0f}GB) — MAX config")

elif "t4" in gpu_name:
    # T4 (16GB) — Colab Free
    CONFIG = {
        "batch_size": 16,
        "grad_accum_steps": 8,
        "d_model": 512,
        "n_layers": 8,
        "max_steps": 10000,
        "eval_interval": 500,
        "checkpoint_interval": 500,
        "compile": True,
    }
    print("🆓 T4 detected — using memory-optimized config")
elif "v100" in gpu_name:
    CONFIG = {
        "batch_size": 24,
        "grad_accum_steps": 4,
        "d_model": 768,
        "n_layers": 12,
        "max_steps": 15000,
        "eval_interval": 500,
        "checkpoint_interval": 1000,
        "compile": False,  # V100 doesn't support torch.compile well
    }
    print("⚡ V100 detected")
else:
    # Conservative fallback
    CONFIG = {
        "batch_size": 8,
        "grad_accum_steps": 16,
        "d_model": 512,
        "n_layers": 8,
        "max_steps": 5000,
        "eval_interval": 250,
        "checkpoint_interval": 250,
        "compile": False,
    }
    print(f"⚠️ Unknown GPU ({gpu_name}) — using conservative config")

effective_batch = CONFIG["batch_size"] * CONFIG["grad_accum_steps"]
print(f"Effective batch size: {effective_batch}")
print(f"Model: d={CONFIG['d_model']}, L={CONFIG['n_layers']}")

🚀 High-end GPU detected (nvidia rtx pro 6000 blackwell server edition, 102GB) — MAX config
Effective batch size: 128
Model: d=512, L=8


In [4]:
# ============================================================
# Cell 3: Download + Tokenize Data (runs once, saved to Drive)
# ============================================================

import os
import subprocess
import sys

TARGET_SHARDS = 10
existing_shards = 0
if os.path.exists(DRIVE_DATA):
    existing_shards = len([f for f in os.listdir(DRIVE_DATA) if f.endswith('.bin')])
if existing_shards >= TARGET_SHARDS:
    print(f"✅ Data already on Drive ({existing_shards} shards). Skipping download.")
else:
    print(f"📦 Have {existing_shards}/{TARGET_SHARDS} shards. Downloading remaining...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "tqdm", "huggingface_hub"], check=True)
    os.makedirs(DRIVE_DATA, exist_ok=True)

    env = os.environ.copy()
    env["PYTHONPATH"] = DRIVE_PROJECT

    # Use --dataset instead of --config_name which is not supported by the script
    result = subprocess.run([
    sys.executable, "-m", "data.download",
    "--dataset", "HuggingFaceFW/FineWeb-Edu",
    "--output_dir", DRIVE_DATA,
    "--max_shards", "10",
], env=env, capture_output=True, text=True)

    if result.returncode == 0:
        print(result.stdout)
        print("✅ Data saved to Drive!")
    else:
        print("❌ Script failed:")
        print(result.stderr)
        print(result.stdout)

✅ Data already on Drive (10 shards). Skipping download.


In [5]:
# ============================================================
# Cell 4: Training Function with Auto-Resume
# ============================================================

import yaml
import time
import json
from pathlib import Path

def load_base_config():
    """Load base.yaml config."""
    with open("configs/base.yaml") as f:
        return yaml.safe_load(f)

def build_config(variant: str) -> dict:
    """Build training config for a variant, optimized for current GPU."""
    cfg = load_base_config()
    n_heads = CONFIG["d_model"] // 64

    # Model overrides
    cfg["model"]["d_model"] = CONFIG["d_model"]
    cfg["model"]["n_layers"] = CONFIG["n_layers"]
    cfg["model"]["n_heads"] = n_heads
    cfg["model"]["n_kv_heads"] = n_heads
    cfg["model"]["attention_type"] = variant

    # Variant-specific overrides
    if variant == "mqa":
        cfg["model"]["n_kv_heads"] = 1
    elif variant == "gqa":
        cfg["model"]["n_kv_heads"] = max(1, n_heads // 4)
    elif variant == "mla":
        cfg["model"]["d_latent"] = CONFIG["d_model"] // 4
        cfg["model"]["rope_head_dim"] = 64
    elif variant == "moh":
        cfg["model"]["moh_top_k"] = max(2, n_heads // 2)
        cfg["model"]["moh_n_shared"] = 1
    elif variant == "nsa":
        cfg["model"]["nsa_block_size"] = 16
        cfg["model"]["nsa_top_k_blocks"] = 8
        cfg["model"]["nsa_window_size"] = 512

    # Training overrides
    cfg["training"]["batch_size"] = CONFIG["batch_size"]
    cfg["training"]["grad_accum_steps"] = CONFIG["grad_accum_steps"]
    cfg["training"]["max_steps"] = CONFIG["max_steps"]
    cfg["training"]["eval_interval"] = CONFIG["eval_interval"]
    cfg["training"]["checkpoint_interval"] = CONFIG["checkpoint_interval"]

    # Data
    cfg["data"]["data_dir"] = DRIVE_DATA
    cfg["data"]["num_workers"] = 8

    # Infrastructure — save to Drive!
    cfg["infra"]["device"] = "cuda"
    cfg["infra"]["dtype"] = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    cfg["infra"]["compile"] = CONFIG["compile"]
    cfg["infra"]["checkpoint_dir"] = f"{DRIVE_CHECKPOINTS}/{variant}"
    cfg["infra"]["log_dir"] = f"{DRIVE_PROJECT}/logs/{variant}"
    cfg["infra"]["wandb_enabled"] = False  # Set True if you have W&B

    return cfg


def check_training_status(variant: str) -> dict:
    """Check if a variant has already been trained (fully or partially)."""
    ckpt_dir = Path(f"{DRIVE_CHECKPOINTS}/{variant}")
    if not ckpt_dir.exists():
        return {"status": "not_started", "step": 0}

    checkpoints = sorted(ckpt_dir.glob("checkpoint_step_*.pt"))
    if not checkpoints:
        return {"status": "not_started", "step": 0}

    latest = checkpoints[-1]
    step = int(latest.stem.split("_")[-1])

    if step >= CONFIG["max_steps"]:
        return {"status": "complete", "step": step}
    else:
        return {"status": "in_progress", "step": step}


def train_variant(variant: str):
    """Train a single variant with auto-resume."""
    from training.trainer import Trainer

    status = check_training_status(variant)
    if status["status"] == "complete":
        print(f"✅ {variant.upper()} already complete (step {status['step']}). Skipping.")
        return

    if status["status"] == "in_progress":
        print(f"🔄 {variant.upper()} resuming from step {status['step']}...")
    else:
        print(f"🆕 {variant.upper()} starting fresh...")

    cfg = build_config(variant)
    trainer = Trainer(cfg)
    trainer.train()

    print(f"✅ {variant.upper()} training complete!")

In [6]:
# ============================================================
# Cell 5: Train All Variants Sequentially
# ============================================================
# Run this cell. If Colab disconnects, just re-run — it auto-resumes.

import shutil
# Copy data from Drive to local SSD (MUCH faster I/O)
LOCAL_DATA = "/content/data/tokenized"
if not os.path.exists(LOCAL_DATA) or len(os.listdir(LOCAL_DATA)) == 0:
    print("📋 Copying data from Drive to local SSD...")
    os.makedirs(LOCAL_DATA, exist_ok=True)
    for f in os.listdir(DRIVE_DATA):
        if f.endswith('.bin'):
            shutil.copy2(os.path.join(DRIVE_DATA, f), LOCAL_DATA)
    print(f"✅ Copied {len(os.listdir(LOCAL_DATA))} shards to local SSD")
else:
    print(f"✅ Data already on local SSD ({len(os.listdir(LOCAL_DATA))} files)")
# Override DRIVE_DATA to use local copy
DRIVE_DATA = LOCAL_DATA


VARIANTS = ["mha", "gqa", "mqa", "swa", "diff", "mla", "moh", "nsa"]

print("=" * 60)
print("TRAINING STATUS")
print("=" * 60)
for v in VARIANTS:
    s = check_training_status(v)
    emoji = {"not_started": "⬜", "in_progress": "🟡", "complete": "🟢"}[s["status"]]
    print(f"  {emoji} {v:>5s}: {s['status']:>12s} (step {s['step']})")
print("=" * 60)

import traceback

for variant in VARIANTS:
    try:
        train_variant(variant)
    except Exception as e:
        print(f"❌ {variant.upper()} failed: {e}")
        traceback.print_exc()
        print("Continuing to next variant...")
        continue


📋 Copying data from Drive to local SSD...
✅ Copied 10 shards to local SSD
TRAINING STATUS
  🟢   mha:     complete (step 5000)
  🟢   gqa:     complete (step 5000)
  🟢   mqa:     complete (step 5000)
  🟢   swa:     complete (step 5000)
  🟢  diff:     complete (step 5000)
  🟢   mla:     complete (step 5000)
  🟢   moh:     complete (step 5000)
  🟢   nsa:     complete (step 5000)
✅ MHA already complete (step 5000). Skipping.
✅ GQA already complete (step 5000). Skipping.
✅ MQA already complete (step 5000). Skipping.
✅ SWA already complete (step 5000). Skipping.
✅ DIFF already complete (step 5000). Skipping.
✅ MLA already complete (step 5000). Skipping.
✅ MOH already complete (step 5000). Skipping.
✅ NSA already complete (step 5000). Skipping.


In [7]:
# Cell 4.5: Train DIFF and NSA (memory-hungry variants)
import gc

HEAVY_VARIANTS = {"diff": 16, "nsa": 16}  # smaller micro-batch

for variant, micro_bs in HEAVY_VARIANTS.items():
    gc.collect()
    torch.cuda.empty_cache()

    status = check_training_status(variant)
    if status["status"] == "complete":
        print(f"✅ {variant.upper()} already done (step {status['step']}). Skipping.")
        continue

    print(f"\n🆕 {variant.upper()} starting (micro_batch={micro_bs}, effective=128)...")
    cfg = build_config(variant)
    cfg["training"]["batch_size"] = micro_bs
    cfg["training"]["grad_accum_steps"] = 128 // micro_bs  # keep effective=128

    from training.trainer import Trainer
    trainer = Trainer(cfg)
    trainer.train()
    print(f"✅ {variant.upper()} complete!")

    del trainer
    gc.collect()
    torch.cuda.empty_cache()


✅ DIFF already done (step 5000). Skipping.
✅ NSA already done (step 5000). Skipping.


In [9]:
# ============================================================
# Cell 6: Profile All Variants
# ============================================================

# Cell 6: Throughput Profiling (small batch, measurement only)
import gc, subprocess, sys
gc.collect()
torch.cuda.empty_cache()

os.chdir(DRIVE_PROJECT)
result = subprocess.run([
    sys.executable, "scripts/profile.py",
    "--device", "cuda",
    "--d_model", "512",
    "--n_layers", "8",
    "--batch_size", "4",
    "--seq_len", "1024",
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("stderr:", result.stderr[-2000:])


Profiling on cuda | d=512 L=8 B=4 T=1024

--- MHA ---
  Step: 55.9ms | 73,250 tok/s | Mem: 6108MB | Params: 59.3M

--- GQA ---
  Step: 54.9ms | 74,636 tok/s | Mem: 6096MB | Params: 56.1M

--- MQA ---
  Step: 52.6ms | 77,855 tok/s | Mem: 6094MB | Params: 55.6M

--- SWA ---
  Step: 54.7ms | 74,856 tok/s | Mem: 6108MB | Params: 59.3M

--- DIFF ---
  Step: 81.9ms | 49,983 tok/s | Mem: 9253MB | Params: 59.3M

--- MLA ---
  Step: 64.7ms | 63,346 tok/s | Mem: 6408MB | Params: 62.4M

--- MOH ---
  Step: 55.9ms | 73,294 tok/s | Mem: 6176MB | Params: 59.3M

--- NSA ---
  Step: 60.4ms | 67,849 tok/s | Mem: 6781MB | Params: 59.4M

 Variant |  ms/step |      tok/s |   Memory |   Params
-------------------------------------------------------
     mqa |    52.6ms |     77,855 |   6094MB |   55.6M
     swa |    54.7ms |     74,856 |   6108MB |   59.3M
     gqa |    54.9ms |     74,636 |   6096MB |   56.1M
     moh |    55.9ms |     73,294 |   6176MB |   59.3M
     mha |    55.9ms |     73,250 |   6108

In [11]:
# Cell 7: Proper Evaluation + Comparison
import gc, json, math
from pathlib import Path
from model import ModelConfig, Transformer
from model.utils import count_parameters
from eval.perplexity import evaluate_perplexity
from eval.compare import RunMetrics, estimate_kv_cache_per_token, format_comparison_table, save_results
from data.dataloader import create_dataloader

VARIANTS_ALL = ["mha","gqa","mqa","swa","diff","mla","moh","nsa"]
profiling_data = {
    "mha": (73250, 6108), "gqa": (74636, 6096), "mqa": (77855, 6094),
    "swa": (74856, 6108), "diff": (49983, 9253), "mla": (63346, 6408),
    "moh": (73294, 6176), "nsa": (67849, 6781),
}

all_metrics = []
n_heads = CONFIG["d_model"] // 64

for variant in VARIANTS_ALL:
    gc.collect(); torch.cuda.empty_cache()
    print(f"\n--- Evaluating {variant.upper()} ---")

    ckpt_dir = Path(f"{DRIVE_CHECKPOINTS}/{variant}")
    ckpts = sorted(ckpt_dir.glob("checkpoint_step_*.pt"))
    if not ckpts:
        print(f"  ⚠️ No checkpoint for {variant}, skipping"); continue

    ckpt = torch.load(ckpts[-1], map_location="cuda", weights_only=False)
    step = ckpt.get("step", 0)

    # Build model config
    kw = {"attention_type": variant, "n_kv_heads": n_heads}
    if variant == "mqa": kw["n_kv_heads"] = 1
    elif variant == "gqa": kw["n_kv_heads"] = max(1, n_heads // 4)
    elif variant == "mla": kw["d_latent"] = CONFIG["d_model"]//4; kw["rope_head_dim"] = 64
    elif variant == "moh": kw["moh_top_k"] = max(2, n_heads//2); kw["moh_n_shared"] = 1
    elif variant == "nsa": kw["nsa_block_size"] = 16; kw["nsa_top_k_blocks"] = 8; kw["nsa_window_size"] = 512

    mcfg = ModelConfig(d_model=CONFIG["d_model"], n_layers=CONFIG["n_layers"], n_heads=n_heads, **kw)
    model = Transformer(mcfg).to("cuda")
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # Evaluate perplexity
    try:
        ppl = evaluate_perplexity(model, DRIVE_DATA, 1024, 16, eval_steps=100, device="cuda")
        val_loss = math.log(ppl)
    except Exception as e:
        print(f"  ⚠️ Eval failed: {e}"); ppl = 0.0; val_loss = 0.0

    tok_s, mem = profiling_data.get(variant, (0, 0))
    params = count_parameters(model)

    m = RunMetrics(
        variant=variant, val_loss=val_loss, val_ppl=ppl,
        tokens_per_sec=tok_s, peak_memory_mb=mem,
        total_params=params.get("total", 0),
        kv_cache_bytes_per_token=estimate_kv_cache_per_token(mcfg),
        final_step=step,
    )
    all_metrics.append(m)
    print(f"  ✅ {variant.upper()}: PPL={ppl:.2f} | {tok_s:,} tok/s | {mem}MB")

    del model, ckpt; gc.collect(); torch.cuda.empty_cache()

save_results(all_metrics, f"{DRIVE_PROJECT}/results")



--- Evaluating MHA ---
ShardedTokenDataset: 10 shards, 1,000,000,000 tokens, 999,998,975 valid positions
  ✅ MHA: PPL=61.97 | 73,250 tok/s | 6108MB

--- Evaluating GQA ---
ShardedTokenDataset: 10 shards, 1,000,000,000 tokens, 999,998,975 valid positions
  ✅ GQA: PPL=63.81 | 74,636 tok/s | 6096MB

--- Evaluating MQA ---
ShardedTokenDataset: 10 shards, 1,000,000,000 tokens, 999,998,975 valid positions
  ✅ MQA: PPL=65.21 | 77,855 tok/s | 6094MB

--- Evaluating SWA ---
ShardedTokenDataset: 10 shards, 1,000,000,000 tokens, 999,998,975 valid positions
  ✅ SWA: PPL=62.74 | 74,856 tok/s | 6108MB

--- Evaluating DIFF ---
ShardedTokenDataset: 10 shards, 1,000,000,000 tokens, 999,998,975 valid positions


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)


  ✅ DIFF: PPL=71.27 | 49,983 tok/s | 9253MB

--- Evaluating MLA ---
ShardedTokenDataset: 10 shards, 1,000,000,000 tokens, 999,998,975 valid positions
  ✅ MLA: PPL=71.40 | 63,346 tok/s | 6408MB

--- Evaluating MOH ---
ShardedTokenDataset: 10 shards, 1,000,000,000 tokens, 999,998,975 valid positions
  ✅ MOH: PPL=69.22 | 73,294 tok/s | 6176MB

--- Evaluating NSA ---
ShardedTokenDataset: 10 shards, 1,000,000,000 tokens, 999,998,975 valid positions
  ✅ NSA: PPL=1.01 | 67,849 tok/s | 6781MB
[Compare] Results saved to /content/drive/MyDrive/minilm-bench/results
| Variant | Val PPL ↓ | Tokens/s ↑ | Peak Mem ↓ | MFU ↑ | KV$/tok ↓ | Params |
|---------|----------|-----------|-----------|-------|----------|--------|
| nsa      |      1.0 |    67,849 |   6,781 MB |  0.0% |   2048 B |  59.4M |
| mha      |     62.0 |    73,250 |   6,108 MB |  0.0% |   2048 B |  59.3M |
| swa      |     62.7 |    74,856 |   6,108 MB |  0.0% |   2048 B |  59.3M |
| gqa      |     63.8 |    74,636 |   6,096 MB |  0.0%